# urban-energy-core Capabilities Walkthrough

This notebook demonstrates the package as a standalone workflow.

For now, it points to a local external data root that contains the current project data.

## 1. Setup

In [1]:
from pathlib import Path
import sys
import warnings

from tqdm import TqdmWarning
import pandas as pd

warnings.filterwarnings("ignore", category=TqdmWarning)
warnings.filterwarnings("ignore", message=r".*IProgress not found.*")
warnings.filterwarnings("ignore", category=DeprecationWarning, message=r".*unary_union.*deprecated.*")

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd().resolve()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from urban_energy_core.io import (
    load_all_fsa_census,
    load_city_fsa_geojsons,
    load_city_weather_csvs,
    load_processed_electricity_wide,
)
from urban_energy_core.pipelines import build_cities_from_data


## 2. Configure Data Root

By default this notebook uses the current external data root on disk.
You can change `DATA_ROOT` later without changing the package itself.

In [2]:
DATA_ROOT = REPO_ROOT.parent / "urban-energy-data"
ELEC_PATH = DATA_ROOT / "data" / "processed" / "electricity" / "elec_rebuilt_new_weather.parquet"
CENSUS_ROOT = DATA_ROOT / "data" / "raw" / "census" / "FSA scale"
GEOMETRY_ROOT = DATA_ROOT / "data" / "raw" / "geometry"
WEATHER_ROOT = DATA_ROOT / "data" / "raw" / "weather"

availability_df = pd.DataFrame(
    {
        "input": [
            "shared_data_root",
            "fsa_electricity",
            "fsa_census",
            "geometry",
            "weather",
        ],
        "available": [
            DATA_ROOT.exists(),
            ELEC_PATH.exists(),
            CENSUS_ROOT.exists(),
            GEOMETRY_ROOT.exists(),
            WEATHER_ROOT.exists(),
        ],
    }
)
availability_df


,input,available
0,shared_data_root,True
1,fsa_electricity,True
2,fsa_census,True
3,geometry,True
4,weather,True


## 3. Load Inputs

In [3]:
elec_df = load_processed_electricity_wide(ELEC_PATH)
census_df = load_all_fsa_census(root_dir=CENSUS_ROOT, drop_key_col=False, show_progress=False)
if "GEO UID" in census_df.columns:
    census_df = census_df.set_index("GEO UID")
census_df.index = census_df.index.astype(str)

geo = load_city_fsa_geojsons(geometry_dir=GEOMETRY_ROOT, show_progress=False)
weather = load_city_weather_csvs(weather_dir=WEATHER_ROOT, show_progress=False)

print("Electricity shape:", elec_df.shape)
print("Census shape:", census_df.shape)
print("Cities in geometry:", sorted(geo.keys()))
print("Cities in weather:", sorted(weather.keys()))

Electricity shape: (140256, 133)
Census shape: (1646, 1122)
Cities in geometry: ['montreal', 'quebec_city', 'trois_rivieres']
Cities in weather: ['montreal', 'quebec_city', 'trois_rivieres']


## 4. Build City Objects

In [4]:
cities = build_cities_from_data(
    elec_df=elec_df,
    city_geojsons=geo,
    city_weather=weather,
    census_df=census_df,
    show_progress=False,
)

sorted(cities.keys())

['montreal', 'quebec_city', 'trois_rivieres']

## 5. Inspect Montreal

In [5]:
montreal = cities["montreal"]
first_fsa_code = montreal.list_fsa_codes()[0]
first_fsa = montreal.get_fsa(first_fsa_code)

demo_fsa_codes = montreal.list_fsa_codes()[:10]
demo_geo = geo["montreal"].loc[geo["montreal"]["FSA"].astype(str).isin(demo_fsa_codes)].copy()
demo_census = census_df.loc[census_df.index.intersection(demo_fsa_codes)].copy()
demo_elec = elec_df.loc[:, demo_fsa_codes].copy()
montreal_demo = build_cities_from_data(
    elec_df=demo_elec,
    city_geojsons={"montreal": demo_geo},
    city_weather={"montreal": weather["montreal"]},
    census_df=demo_census,
    show_progress=False,
)["montreal"]

{
    "montreal_fsa_count": len(montreal.fsas),
    "demo_fsa_count": len(montreal_demo.fsas),
    "example_fsa": first_fsa_code,
    "electricity_frame_shape": montreal.electricity_frame().shape,
}


{'montreal_fsa_count': 94,
 'demo_fsa_count': 10,
 'example_fsa': 'H1A',
 'electricity_frame_shape': (140256, 94)}

## 6. Per-FSA Operations

In [6]:
normalized = first_fsa.normalize_for_weather(montreal.weather)
per_capita = first_fsa.per_capita_consumption()
heating_prism = first_fsa.apply_heating_prism(montreal.weather)
short_term_daily = first_fsa.short_term_metrics(show_progress=False)

print("Normalized length:", len(normalized))
print("Per-capita mean:", float(per_capita.mean()))
print("Heating change point:", heating_prism["heating_change_point_temp_c"])
print("Short-term rows:", len(short_term_daily))

Normalized length: 140256
Per-capita mean: 0.4007574887069917
Heating change point: 8.0
Short-term rows: 1461


In [7]:
short_term_daily.head()

,peak_load,p90_top10_mean,am_pm_peak_ratio,ramp_up_rate,dtw_cluster_label
date,,,,,
2019-01-01 00:00:00-05:00,16280.393381,15614.007845,0.885711,636.860625,NaN
2019-01-02 00:00:00-05:00,20327.246500,19985.976375,0.760900,763.630664,NaN
2019-01-03 00:00:00-05:00,20909.136582,20649.993329,0.819962,1218.418907,NaN
2019-01-04 00:00:00-05:00,18581.843775,18218.320693,0.953652,749.929570,NaN
2019-01-05 00:00:00-05:00,17114.224634,16794.948595,0.955262,750.639610,NaN


## 7. City-Level Tables

In [8]:
prism_table = montreal_demo.compute_prism_table(show_progress=False)
short_term_table = montreal_demo.compute_short_term_table(show_progress=False)

print("PRISM table shape:", prism_table.shape)
print("Short-term table shape:", short_term_table.shape)


PRISM table shape: (10, 22)
Short-term table shape: (10, 5)


In [9]:
prism_table[["heating_slope_per_hdd", "r2"]].head(10)

,heating_slope_per_hdd,r2
fsa,,
H1E,0.0096,0.542533
H1C,0.0090,0.560993
H1B,0.0085,0.530845
H1A,0.0082,0.526389
H1L,0.0080,0.508394
H1M,0.0079,0.487929
H1K,0.0073,0.511952
H1J,0.0072,0.494025
H1H,0.0064,0.460950


In [10]:
short_term_table.head(10)

,peak_load,p90_top10_mean,am_pm_peak_ratio,ramp_up_rate,dtw_cluster_label
fsa,,,,,
H1A,0.521219,0.510542,0.868295,0.021648,NaN
H1B,0.559643,0.548195,0.863818,0.024338,NaN
H1C,0.524899,0.514288,0.883234,0.022965,NaN
H1E,0.592264,0.581262,0.902065,0.021967,NaN
H1G,0.454229,0.446316,0.885692,0.016854,NaN
H1H,0.480241,0.471685,0.891102,0.017869,NaN
H1J,0.473608,0.462121,0.886201,0.023722,NaN
H1K,0.470673,0.461800,0.886149,0.020128,NaN
H1L,0.531571,0.521524,0.873519,0.020441,NaN


## 8. Map Object

In [11]:
map_fig = montreal_demo.plot_map(metric="mean", alpha=0.4, figsize=(8, 5), show=False)
map_fig


## 9. Example Interpretation

In [12]:
strongest = prism_table["heating_slope_per_hdd"].idxmax()
strongest_row = prism_table.loc[strongest]

print(
    f"Highest heating-sensitive Montreal FSA: {strongest} "
    f"(slope={strongest_row['heating_slope_per_hdd']:.4f}, r2={strongest_row['r2']:.3f})"
)

Highest heating-sensitive Montreal FSA: H1E (slope=0.0096, r2=0.543)


## 10. Notes

- This notebook presents `urban-energy-core` as a standalone package.
- The current `DATA_ROOT` is just the local source of data for now.
- Later, the same notebook can point at package-native data locations or a different external root.